In [ ]:
!pip install --upgrade transformers --quiet
!pip install onnx onnxruntime-gpu --quiet
!pip install onnxscript --quiet

import os, time, json, random, warnings
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader, Sampler
import onnx

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================
ROOT = "/content/drive/Shareddrives/Garment Type/Complete_dataset"
OUTDIR = Path("/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub")
OUTDIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ============================================================
# KEY DESIGN DECISIONS
# ============================================================
# 1. Using SigLIP2-SO400M vision encoder (same as macro router)
# 2. This replaces the old SigLIP sub-classifier AND the CLIP probe
#    — SigLIP2 provides the semantic/ViT signal
#    — ConvNeXt-base provides the CNN/texture signal
#    — Together they form a CNN+ViT ensemble
# 3. jogging_bottoms gets class weight boost (known hard class)
# 4. ONNX export with softmax baked in for TRT
# ============================================================

# MODEL
SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
HIDDEN_SIZE = 1152  # SigLIP2-SO400M pooler output dimension

# TRAINING PARAMS
BATCH = 16              # 428M param model
EPOCHS = 60
HEAD_EPOCHS = 10

LR_HEAD = 5e-4          # Stage 1
BACKBONE_LR = 2e-6      # Stage 2: very gentle
HEAD_LR = 1e-4           # Stage 2
WEIGHT_DECAY = 1e-2

PATIENCE = 12
WARMUP_EPOCHS = 3
NUM_WORKERS = 2

MIN_SAMPLES_FOR_BALANCED = 12

# LOSS SETTINGS
LABEL_SMOOTH = 0.05
FOCAL_GAMMA = 2.0

# MIXUP
MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.3

# VAL SPLIT
VAL_RATIO = 0.16

# NORMALIZATION — SigLIP2 uses [0.5, 0.5, 0.5]
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

LOWER_CLASSES = ["jeans", "jogging_bottoms", "skirts", "trousers"]
MINORITY_CLASSES = {"skirts"}
MEDIUM_CLASSES = {"jeans"}


# ============================================================
# MODEL DEFINITION
# ============================================================
class SigLIP2Classifier(nn.Module):
    """SigLIP2 vision encoder + classifier head for garment sub-classification."""

    def __init__(self, model_id, num_classes, dropout_rate=0.3):
        super().__init__()

        # Load SigLIP2 vision encoder
        # Prioritize AutoModel for robustness
        try:
            from transformers import AutoModel
            full_model = AutoModel.from_pretrained(model_id, ignore_mismatched_sizes=True)
            # AutoModel will instantiate the correct model type (e.g., Siglip2ForImageClassification, etc.)
            # We need the vision component from it.
            self.vision = full_model.vision_model if hasattr(full_model, 'vision_model') else full_model

        except Exception as e_auto:
            print(f"AutoModel loading failed: {e_auto}. Falling back to direct Siglip2VisionModel.")
            # Fallback to direct Siglip2VisionModel if AutoModel fails
            try:
                from transformers import Siglip2VisionModel
                self.vision = Siglip2VisionModel.from_pretrained(model_id, ignore_mismatched_sizes=True)
            except ImportError:
                try:
                    from transformers import SiglipVisionModel
                    self.vision = SiglipVisionModel.from_pretrained(model_id, ignore_mismatched_sizes=True)
                except ImportError as e_siglip:
                    raise ImportError(f"Could not load any SigLIP vision model: {e_siglip}") from e_siglip

        # DEBUG: Print patch_embedding details
        if hasattr(self.vision, 'embeddings') and hasattr(self.vision.embeddings, 'patch_embedding'):
            pe = self.vision.embeddings.patch_embedding
            print(f"DEBUG: Type of patch_embedding: {type(pe)}")
            if isinstance(pe, nn.Linear):
                print(f"DEBUG: patch_embedding is Linear. in_features: {pe.in_features}, out_features: {pe.out_features}")
            elif isinstance(pe, nn.Conv2d):
                print(f"DEBUG: patch_embedding is Conv2d. in_channels: {pe.in_channels}, out_channels: {pe.out_channels}, kernel_size: {pe.kernel_size}, stride: {pe.stride}")
            else:
                print(f"DEBUG: patch_embedding is unknown type: {type(pe)}")

        # Classifier head
        self.classifier = nn.Sequential(
            nn.LayerNorm(HIDDEN_SIZE),
            nn.Dropout(dropout_rate),
            nn.Linear(HIDDEN_SIZE, num_classes),
        )

        self._num_classes = num_classes

    def forward(self, pixel_values):
        batch_size = pixel_values.shape[0]
        # Based on SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
        # Patch size is 14, Image size is 384. 384 // 14 = 27.
        # This suggests a patch grid of 27x27 = 729 patches.
        # The pixel_attention_mask should be for these patches.
        num_patches_per_dim = IMG_SIZE // 14 # Assuming 14 is the patch size mentioned in model_id
        num_patches_flat = num_patches_per_dim * num_patches_per_dim # 27 * 27 = 729

        # Create dummy pixel_attention_mask (all ones for full image)
        pixel_attention_mask = torch.ones(
            (batch_size, num_patches_flat),
            dtype=torch.long, device=pixel_values.device
        )

        # Create dummy spatial_shapes
        spatial_shapes = torch.tensor(
            [[num_patches_per_dim, num_patches_per_dim]],
            dtype=torch.long, device=pixel_values.device
        )

        # Pass all required arguments to the vision model's forward method
        outputs = self.vision(
            pixel_values=pixel_values,
            pixel_attention_mask=pixel_attention_mask,
            spatial_shapes=spatial_shapes
        )
        pooled = outputs.pooler_output
        return self.classifier(pooled)

    def get_vision_params(self):
        return list(self.vision.parameters())

    def get_head_params(self):
        return list(self.classifier.parameters())


class SigLIP2ForExport(nn.Module):
    """Wrapper that includes softmax for ONNX/TRT export."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        logits = self.model(pixel_values)
        return F.softmax(logits, dim=1)


# ============================================================
# FOCAL LOSS + LABEL SMOOTHING
# ============================================================
def focal_loss_with_ls(logits, targets, gamma=2.0, smooth=0.05, weight=None):
    num_classes = logits.shape[1]
    one_hot = F.one_hot(targets, num_classes).float().to(logits.device)
    one_hot = one_hot * (1 - smooth) + smooth / num_classes

    log_probs = F.log_softmax(logits, dim=1)
    probs = torch.exp(log_probs)

    focal_factor = (1 - probs) ** gamma
    loss = -one_hot * focal_factor * log_probs
    loss = loss.sum(dim=1)

    if weight is not None:
        loss = loss * weight[targets]

    return loss.mean()


def focal_loss_mixup(logits, targets_a, targets_b, lam, gamma=2.0, smooth=0.05, weight=None):
    loss_a = focal_loss_with_ls(logits, targets_a, gamma, smooth, weight)
    loss_b = focal_loss_with_ls(logits, targets_b, gamma, smooth, weight)
    return lam * loss_a + (1 - lam) * loss_b


# ============================================================
# MIXUP
# ============================================================
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    lam = max(lam, 1 - lam)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


# ============================================================
# UTILITIES
# ============================================================
def collect_samples(root, allowed_classes):
    root = Path(root)
    samples = []
    for dataset_dir in root.iterdir():
        if not dataset_dir.is_dir():
            continue
        for cls_dir in dataset_dir.iterdir():
            if not cls_dir.is_dir():
                continue
            cls_name = cls_dir.name.lower()
            if cls_name not in allowed_classes:
                continue
            for img_path in cls_dir.iterdir():
                if img_path.is_file():
                    samples.append((str(img_path), cls_name))
    return samples


# ============================================================
# AUGMENTATIONS — SigLIP2 normalization
# ============================================================
def build_aug(img_size, is_train=True):
    h = w = img_size

    if not is_train:
        return A.Compose([
            A.Resize(h, w),
            A.Normalize(NORM_MEAN, NORM_STD),
            ToTensorV2()
        ])

    return A.Compose([
        A.RandomResizedCrop(size=(h, w), scale=(0.75, 1.0), ratio=(0.75, 1.3), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.04, scale_limit=0.08, rotate_limit=8, p=0.6),
        A.ColorJitter(0.2, 0.2, 0.2, 0.05, p=0.6),
        A.GaussianBlur(3, p=0.2),
        A.CoarseDropout(max_holes=4, max_height=int(h * 0.10), max_width=int(w * 0.10), p=0.3),
        A.Normalize(NORM_MEAN, NORM_STD),
        ToTensorV2()
    ])


def build_strong_aug(img_size):
    h = w = img_size
    return A.Compose([
        A.RandomResizedCrop(size=(h, w), scale=(0.60, 1.0), ratio=(0.70, 1.4), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.12, rotate_limit=12, p=0.8),
        A.ColorJitter(0.25, 0.25, 0.25, 0.08, p=0.8),
        A.GaussianBlur(blur_limit=3, p=0.3),
        A.CoarseDropout(max_holes=6, max_height=int(h * 0.12), max_width=int(w * 0.12), p=0.4),
        A.Normalize(NORM_MEAN, NORM_STD),
        ToTensorV2()
    ])


# ============================================================
# DATASET
# ============================================================
class ImgDataset(Dataset):
    def __init__(self, samples, cls2idx, img_size, aug=True):
        self.samples = samples
        self.cls2idx = cls2idx
        self.img_size = img_size
        self.aug = aug
        self.base_aug = build_aug(img_size, is_train=aug)
        self.strong_aug = build_strong_aug(img_size)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        p, c = self.samples[i]
        try:
            img = np.array(Image.open(p).convert("RGB"))
        except:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)

        if self.aug and c in MINORITY_CLASSES:
            data = self.strong_aug(image=img)
        elif self.aug and c in MEDIUM_CLASSES:
            if random.random() < 0.5:
                data = self.strong_aug(image=img)
            else:
                data = self.base_aug(image=img)
        else:
            data = self.base_aug(image=img)

        return data["image"], self.cls2idx[c], p


# ============================================================
# BALANCED SAMPLER
# ============================================================
class BalancedBatchSampler(Sampler):
    def __init__(self, labels, n_cls, n_per):
        self.labels = labels
        self.n_cls = n_cls
        self.n_per = n_per
        self.lab2idx = defaultdict(list)
        for i, l in enumerate(labels):
            self.lab2idx[l].append(i)
        for l in self.lab2idx:
            random.shuffle(self.lab2idx[l])
        self.classes = list(self.lab2idx.keys())
        self.used = {c: 0 for c in self.classes}
        self.num_batches = sum(len(v) for v in self.lab2idx.values()) // (n_cls * n_per)

    def __iter__(self):
        for _ in range(self.num_batches):
            chosen = random.sample(self.classes, self.n_cls)
            batch = []
            for c in chosen:
                st = self.used[c]
                en = st + self.n_per
                if en > len(self.lab2idx[c]):
                    random.shuffle(self.lab2idx[c])
                    st = 0; en = self.n_per
                batch.extend(self.lab2idx[c][st:en])
                self.used[c] = en
            yield batch

    def __len__(self):
        return self.num_batches


# ============================================================
# LOAD DATA
# ============================================================
print("Scanning:", ROOT)
samples = collect_samples(ROOT, set(LOWER_CLASSES))

class_counts = Counter(c for _, c in samples)
print("\n=== TOTAL IMAGE COUNT PER CLASS ===")
for cls, cnt in class_counts.most_common():
    print(f"  {cls:20s} : {cnt}")
print(f"  {'TOTAL':20s} : {len(samples)}")

# Train/Val split — stratified
by_class = defaultdict(list)
for p, c in samples:
    by_class[c].append((p, c))

train_samples = []
val_samples = []
rng = random.Random(SEED)

for c, arr in by_class.items():
    arr = list(arr)
    rng.shuffle(arr)
    nval = max(2, int(len(arr) * VAL_RATIO))
    val_samples += arr[:nval]
    train_samples += arr[nval:]

print(f"\nTrain: {len(train_samples)}  Val: {len(val_samples)}")

# Save val split
val_split_info = [{"path": p, "label": c} for p, c in val_samples]
val_split_path = OUTDIR / "val_split.json"
with open(val_split_path, "w") as f:
    json.dump(val_split_info, f, indent=2)
print(f"Val split saved to: {val_split_path}")


# ============================================================
# TRAINING
# ============================================================
def train_lower():
    cls2idx = {c: i for i, c in enumerate(sorted(LOWER_CLASSES))}
    idx2cls = {v: k for k, v in cls2idx.items()}
    print(f"\nClasses: {cls2idx}")

    tr = train_samples
    vl = val_samples
    cnt = Counter([c for _, c in tr])
    print("Train counts:", dict(cnt))

    labels = [c for _, c in tr]

    # --- Balanced sampler ---
    if all(cnt[c] >= MIN_SAMPLES_FOR_BALANCED for c in cnt):
        ncls = min(len(LOWER_CLASSES), 4)
        nper = max(2, BATCH // ncls)
        sampler = BalancedBatchSampler(labels, ncls, nper)
        tr_ds = ImgDataset(tr, cls2idx, IMG_SIZE, aug=True)
        tr_loader = DataLoader(tr_ds, batch_sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
    else:
        tr_ds = ImgDataset(tr, cls2idx, IMG_SIZE, aug=True)
        weights = [1 / (cnt[c] + 1e-6) for c in labels]
        sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights))
        tr_loader = DataLoader(tr_ds, batch_size=BATCH, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)

    vl_ds = ImgDataset(vl, cls2idx, IMG_SIZE, aug=False)
    vl_loader = DataLoader(vl_ds, batch_size=24, shuffle=False, num_workers=2)

    # ============================================================
    # CLASS WEIGHTS
    # ============================================================
    class_w = torch.ones(len(cls2idx), device=DEVICE)
    for cls, idx in cls2idx.items():
        n = cnt.get(cls, 0)
        if n > 0:
            class_w[idx] = (1 - 0.9999) / (1 - (0.9999 ** n))
        else:
            class_w[idx] = 1.0
    class_w = class_w / class_w.mean()

    # Lower-specific boosts
    boost = {
        "skirts": 1.8,           # minority class
        "jogging_bottoms": 1.4,  # known hard class (confuses with trousers)
    }
    for cls, factor in boost.items():
        if cls in cls2idx:
            class_w[cls2idx[cls]] *= factor
    class_w = class_w / class_w.mean()

    print(f"\nClass weights:")
    for cls in sorted(cls2idx.keys()):
        print(f"  {cls:20s}: {class_w[cls2idx[cls]]:.4f}")

    # ============================================================
    # CREATE MODEL
# ============================================================
    print(f"\nLoading SigLIP2: {SIGLIP2_MODEL_ID}")
    model = SigLIP2Classifier(SIGLIP2_MODEL_ID, len(cls2idx), dropout_rate=0.3).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    vision_params = sum(p.numel() for p in model.vision.parameters())
    head_params_count = sum(p.numel() for p in model.classifier.parameters())
    print(f"Total: {total_params / 1e6:.1f}M  Vision: {vision_params / 1e6:.1f}M  Head: {head_params_count / 1e6:.2f}M")

    scaler = torch.amp.GradScaler("cuda")

    # ============================================================
    # STAGE 1: HEAD-ONLY
    # ============================================================
    print(f"\n--- Stage 1: Head-only training ({HEAD_EPOCHS} epochs) ---")

    for p in model.vision.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True

    opt = torch.optim.AdamW(model.get_head_params(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

    for ep in range(1, HEAD_EPOCHS + 1):
        model.train()
        losses = []
        for xb, yb, _ in tr_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                out = model(xb)
                loss = focal_loss_with_ls(out, yb, gamma=FOCAL_GAMMA, smooth=LABEL_SMOOTH, weight=class_w)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())

        print(f"  [head] ep{ep}/{HEAD_EPOCHS}  loss={np.mean(losses):.4f}")

    # ============================================================
    # STAGE 2: FULL FINE-TUNING
    # ============================================================
    print(f"\n--- Stage 2: Full fine-tuning ({EPOCHS} epochs, MixUp α={MIXUP_ALPHA}) ---")

    for p in model.vision.parameters():
        p.requires_grad = True

    opt = torch.optim.AdamW([
        {"params": model.get_vision_params(), "lr": BACKBONE_LR},
        {"params": model.get_head_params(), "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    sched = torch.optim.lr_scheduler.ReduceLOnPlateau(
        opt, mode="max", factor=0.5, patience=5, min_lr=1e-7
    )

    initial_lrs = [g["lr"] for g in opt.param_groups]
    best_f1 = -1
    patience_counter = 0

    # ============================================================
    # CRASH-SAFE RESUMABLE CHECKPOINT (helpers + resume)
    # ============================================================
    import shutil as _shutil
    LAST_CKPT_NAME = "best_siglip2_lower_last.pt"

    def _atomic_save(obj, final_path):
        tmp = str(final_path) + ".tmp"
        torch.save(obj, tmp)
        os.replace(tmp, final_path)

    def _save_last(state, name, out_dir):
        local = os.path.join("/content", name)
        _atomic_save(state, local)
        drive = os.path.join(str(out_dir), name)
        if drive.endswith(".pt"):     prev = drive[:-3] + "_prev.pt"
        elif drive.endswith(".pth"):  prev = drive[:-4] + "_prev.pth"
        else:                         prev = drive + "_prev"
        try:
            if os.path.exists(drive):
                try: _shutil.copy2(drive, prev)
                except Exception: pass
            tmp = drive + ".tmp"
            _shutil.copy2(local, tmp)
            os.replace(tmp, drive)
        except Exception as e:
            print(f"  [last-ckpt] Drive copy failed (local copy still safe): {e}")

    def _try_load_last(name, out_dir):
        if name.endswith(".pt"):     prev = name[:-3] + "_prev.pt"
        elif name.endswith(".pth"):  prev = name[:-4] + "_prev.pth"
        else:                        prev = name + "_prev"
        for p in [os.path.join("/content", name),
                  os.path.join(str(out_dir), name),
                  os.path.join(str(out_dir), prev)]:
            if not os.path.exists(p): continue
            try:
                ck = torch.load(p, map_location="cpu", weights_only=False)
                _ = ck["epoch"]; _ = ck["model_state"]; _ = ck["optimizer"]
                print(f"  [resume] found checkpoint at {p} (epoch {ck['epoch']})")
                return ck
            except Exception as e:
                print(f"  [resume] corrupt ckpt at {p}: {e}")
        print("  [resume] no resume checkpoint — starting fresh.")
        return None

    # --- Try to resume previous training run ---
    start_ep = 1
    _resume = _try_load_last(LAST_CKPT_NAME, OUTDIR)
    if _resume is not None:
        model.load_state_dict(_resume["model_state"])
        opt.load_state_dict(_resume["optimizer"])
        try: sched.load_state_dict(_resume["scheduler"])
        except Exception as _e: print(f"  [resume] scheduler restore skipped: {_e}")
        try: scaler.load_state_dict(_resume["scaler"])
        except Exception as _e: print(f"  [resume] scaler restore skipped: {_e}")
        start_ep         = _resume["epoch"] + 1
        best_f1          = _resume.get("best_f1", best_f1)
        patience_counter = _resume.get("patience_counter", 0)
        if "rng_torch"  in _resume:
            try: torch.set_rng_state(_resume["rng_torch"])
            except Exception: pass
        if "rng_numpy"  in _resume:
            try: np.random.set_state(_resume["rng_numpy"])
            except Exception: pass
        if "rng_python" in _resume:
            try: random.setstate(_resume["rng_python"])
            except Exception: pass
        if "rng_cuda"   in _resume and torch.cuda.is_available() and _resume["rng_cuda"] is not None:
            try: torch.cuda.set_rng_state_all(_resume["rng_cuda"])
            except Exception: pass
        print(f"  [resume] best_f1={best_f1:.4f}, patience={patience_counter}, next_ep={start_ep}")
        del _resume

    for ep in range(start_ep, EPOCHS + 1):
        model.train()
        losses = []
        n_mixup = 0

        for xb, yb, _ in tr_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                if random.random() < MIXUP_PROB:
                    xb_mixed, ya, yb_mix, lam = mixup_data(xb, yb, alpha=MIXUP_ALPHA)
                    out = model(xb_mixed)
                    loss = focal_loss_mixup(out, ya, yb_mix, lam,
                                            gamma=FOCAL_GAMMA, smooth=LABEL_SMOOTH, weight=class_w)
                    n_mixup += 1
                else:
                    out = model(xb)
                    loss = focal_loss_with_ls(out, yb, gamma=FOCAL_GAMMA, smooth=LABEL_SMOOTH, weight=class_w)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())

        # Warmup
        if ep <= WARMUP_EPOCHS:
            scale = ep / float(WARMUP_EPOCHS)
            for g, init_lr in zip(opt.param_groups, initial_lrs):
                g["lr"] = init_lr * scale

        # ============================================================
        # VALIDATION
        # ============================================================
        model.eval()
        y_true, y_pred = [], []
        all_logits = []

        with torch.no_grad():
            for xb, yb, _ in vl_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    logits = model(xb)
                preds = logits.argmax(1).cpu().numpy()
                y_pred += preds.tolist()
                y_true += yb.numpy().tolist()
                all_logits.append(logits.cpu())

        f1 = f1_score(y_true, y_pred, average="macro")
        acc = accuracy_score(y_true, y_pred)
        cur_lr = opt.param_groups[0]["lr"]

        if ep > WARMUP_EPOCHS:
            old_lr = opt.param_groups[0]["lr"]
            sched.step(f1)
            new_lr = opt.param_groups[0]["lr"]
            if new_lr < old_lr:
                print(f"  ↓ LR reduced: {old_lr:.2e} → {new_lr:.2e}")

        print(f"  [full] ep{ep}/{EPOCHS}  loss={np.mean(losses):.4f}  "
              f"f1={f1:.4f}  acc={acc:.4f}  lr={cur_lr:.2e}  mixup={n_mixup}")

        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0

            torch.save({
                "vision_model": model.vision.state_dict(),
                "classifier": model.classifier.state_dict(),
                "classes": sorted(cls2idx.keys()),
                "cls2idx": cls2idx,
                "branch": "lower",
                "backbone": SIGLIP2_MODEL_ID,
                "hidden_size": HIDDEN_SIZE,
                "img_size": IMG_SIZE,
                "best_f1": best_f1,
                "epoch": ep,
                "normalization": {"mean": list(NORM_MEAN), "std": list(NORM_STD)},
            }, OUTDIR / "best_siglip2_lower.pt")

            # Save val predictions
            all_logits_cat = torch.cat(all_logits, dim=0)
            all_probs = torch.softmax(all_logits_cat, dim=1).numpy()
            np.savez(
                OUTDIR / "val_predictions.npz",
                y_true=np.array(y_true),
                y_pred=np.array(y_pred),
                probs=all_probs,
                classes=sorted(cls2idx.keys())
            )
            print(f"  ✓ Saved BEST (f1={best_f1:.4f}) + val predictions")
        else:
            patience_counter += 1

        # --- Save resumable "last" checkpoint every epoch ---
        _last_state = {
            "epoch": ep,
            "model_state": model.state_dict(),
            "optimizer": opt.state_dict(),
            "scheduler": sched.state_dict(),
            "scaler": scaler.state_dict(),
            "best_f1": best_f1,
            "patience_counter": patience_counter,
            "rng_torch": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "rng_numpy": np.random.get_state(),
            "rng_python": random.getstate(),
        }
        _save_last(_last_state, LAST_CKPT_NAME, OUTDIR)
        del _last_state

        if patience_counter >= PATIENCE:
            print(f"  Early stopping at ep{ep} (patience={PATIENCE})")
            break

    # ============================================================
    # FINAL REPORT
    # ============================================================
    print(f"\n{'='*60}")
    print(f"  TRAINING COMPLETE — Best F1: {best_f1:.4f}")
    print(f"{'='*60}")

    rep = classification_report(y_true, y_pred, target_names=sorted(cls2idx.keys()), digits=4)
    print(rep)
    (OUTDIR / "lower_siglip2_report.txt").write_text(rep)
    np.save(OUTDIR / "lower_siglip2_cm.npy", confusion_matrix(y_true, y_pred))

    # ============================================================
    # COMPUTE THRESHOLDS
# ============================================================
    print("\n--- Computing per-class thresholds ---")
    val_data = np.load(OUTDIR / "val_predictions.npz", allow_pickle=True)
    vt = val_data["y_true"]
    vp = val_data["y_pred"]
    vprobs = val_data["probs"]
    classes = list(val_data["classes"])

    thresholds = {}
    stats = {}
    TARGET_PRECISION = 0.90

    for cls_idx, cls_name in enumerate(classes):
        mask = (vp == cls_idx)
        n_total = mask.sum()
        if n_total < 5:
            thresholds[cls_name] = 0.95
            stats[cls_name] = {"threshold": 0.95, "note": f"Too few ({n_total})"}
            continue

        pred_conf = vprobs[mask, cls_idx]
        pred_correct = (vt[mask] == cls_idx)

        best_t = 0.95
        for t in np.arange(0.20, 0.99, 0.01):
            above = pred_conf >= t
            if above.sum() == 0:
                continue
            precision = pred_correct[above].sum() / above.sum()
            if precision >= TARGET_PRECISION:
                best_t = round(float(t), 2)
                break

        thresholds[cls_name] = best_t
        n_accepted = int((pred_conf >= best_t).sum())
        stats[cls_name] = {
            "threshold": best_t,
            "n_accepted": n_accepted,
            "n_rejected": int(n_total) - n_accepted,
            "n_total": int(n_total),
            "rejection_rate": round((int(n_total) - n_accepted) / max(int(n_total), 1), 4)
        }

    print(f"\n  {'Class':<20s} {'Threshold':>10s} {'Accept':>8s} {'Reject':>8s} {'Rej%':>8s}")
    print(f"  {'-'*55}")
    for cls_name in classes:
        s = stats[cls_name]
        rej = f"{s.get('rejection_rate', 0)*100:.1f}%" if 'rejection_rate' in s else "N/A"
        print(f"  {cls_name:<20s} {s['threshold']:>10.2f} {s.get('n_accepted','N/A'):>8} "
              f"{s.get('n_rejected','N/A'):>8} {rej:>8s}")

    thresh_path = OUTDIR / "lower_siglip2_thresholds.json"
    with open(thresh_path, "w") as f:
        json.dump({"thresholds": thresholds, "calibration_stats": stats}, f, indent=2)

    # ============================================================
    # ONNX EXPORT
    # ============================================================
    print("\n--- Exporting ONNX ---")

    # Reload best checkpoint
    ckpt = torch.load(OUTDIR / "best_siglip2_lower.pt", map_location=DEVICE, weights_only=False)
    model.vision.load_state_dict(ckpt["vision_model"])
    model.classifier.load_state_dict(ckpt["classifier"])
    model.eval()

    export_model = SigLIP2ForExport(model).eval().to(DEVICE)

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    onnx_path = OUTDIR / "siglip2_lower.onnx"

    with torch.no_grad():
        torch.onnx.export(
            export_model,
            dummy,
            str(onnx_path),
            opset_version=18,
            input_names=["pixel_values"],
            output_names=["probabilities"],
            dynamic_axes=None,
        )

    print(f"  ONNX saved: {onnx_path} ({os.path.getsize(onnx_path) / 1e6:.0f} MB)")

    # Verify
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    print("  ✅ ONNX verification passed")

    # Save production config
    config = {
        "model": "SigLIP2_SO400M_Lower",
        "model_id": SIGLIP2_MODEL_ID,
        "branch": "lower",
        "img_size": IMG_SIZE,
        "num_classes": len(cls2idx),
        "class_to_idx": cls2idx,
        "class_names": sorted(cls2idx.keys()),
        "thresholds": thresholds,
        "normalization": {"mean": list(NORM_MEAN), "std": list(NORM_STD)},
        "preprocessing": {
            "normalize_mean": list(NORM_MEAN),
            "normalize_std": list(NORM_STD),
        },
        "training_info": {
            "best_f1": best_f1,
            "backbone": SIGLIP2_MODEL_ID,
            "hidden_size": HIDDEN_SIZE,
        },
        "onnx_output": "probabilities",
        "note": "Output includes softmax — do NOT apply softmax again in production",
    }
    config_path = OUTDIR / "siglip2_lower_config.json"
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"  Config saved: {config_path}")

    print(f"\n{'='*60}")
    print(f"  NEXT STEPS:")
    print(f"  1. Run onnx-simplifier: python -m onnxsim siglip2_lower.onnx siglip2_lower_sim.onnx")
    print(f"  2. Copy to Orin and build TRT engine:")
    print(f"     trtexec --onnx=siglip2_lower_sim.onnx --saveEngine=siglip2_lower_fp16.engine --fp16 --memPoolSize=workspace:8192M")
    print(f"  3. If FP16 fails (attention precision), use FP32:")
    print(f"     trtexec --onnx=siglip2_lower_sim.onnx --saveEngine=siglip2_lower_fp32.engine --memPoolSize=workspace:8192M")
    print(f"{'='*60}")

    return best_f1


# ============================================================
# RUN
# ============================================================
best_f1 = train_lower()

summary = {
    "backbone": SIGLIP2_MODEL_ID,
    "img_size": IMG_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "best_f1": best_f1,
    "mixup_alpha": MIXUP_ALPHA,
    "val_ratio": VAL_RATIO,
}
(OUTDIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"\nDONE. All outputs saved in: {OUTDIR}")


In [ ]:
"""
SigLIP2 Lower Sub-Classifier — Inference & Evaluation
======================================================
Runs on both test datasets:
  1. classified_images_13_14...
  2. Careys_labelled_data

Handles mixed-case folder names and filters only lower classes.
Includes jogging_bottoms ↔ trousers deep dive.
"""

import os, json, random, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = Path("/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub")

# Both test dataset roots — UPDATE paths if needed
TEST_DATASETS = {
    "classified_13_14": "/content/drive/Shareddrives/Garment Type/classified_images_13_14_test",
    "careys": "/content/drive/Shareddrives/Garment Type/Careys_labelled_data",
}

EVAL_DIR = MODEL_DIR / "eval_results"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Model config
SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
HIDDEN_SIZE = 1152
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)

BATCH_SIZE = 24
NUM_WORKERS = 2

# ============================================================
# FOLDER NAME → CANONICAL CLASS MAPPING
# ============================================================
LOWER_FOLDER_MAP = {
    # jeans
    "jeans": "jeans",           "Jeans": "jeans",           "JEANS": "jeans",
    # jogging_bottoms
    "jogging_bottoms": "jogging_bottoms",
    "Jogging_Bottoms": "jogging_bottoms",
    "Jogging_bottoms": "jogging_bottoms",
    "jogging bottoms": "jogging_bottoms",
    "joggers": "jogging_bottoms",
    "Joggers": "jogging_bottoms",
    # skirts
    "skirts": "skirts",         "Skirts": "skirts",         "SKIRTS": "skirts",
    "skirt": "skirts",          "Skirt": "skirts",
    # trousers
    "trousers": "trousers",     "Trousers": "trousers",     "TROUSERS": "trousers",
}

LOWER_CLASSES = sorted(set(LOWER_FOLDER_MAP.values()))


# ============================================================
# MODEL DEFINITION
# ============================================================
class SigLIP2Classifier(nn.Module):
    def __init__(self, model_id, num_classes, dropout_rate=0.3):
        super().__init__()
        from transformers import AutoModel
        full_model = AutoModel.from_pretrained(model_id, ignore_mismatched_sizes=True)
        self.vision = full_model.vision_model
        del full_model

        self.classifier = nn.Sequential(
            nn.LayerNorm(HIDDEN_SIZE),
            nn.Dropout(dropout_rate),
            nn.Linear(HIDDEN_SIZE, num_classes),
        )

    def forward(self, pixel_values):
        outputs = self.vision(pixel_values=pixel_values)
        pooled = outputs.pooler_output
        return self.classifier(pooled)


# ============================================================
# DATA COLLECTION WITH NAME MAPPING
# ============================================================
def collect_test_samples(root):
    """Collect test samples with folder name mapping. Flat: ROOT/class_name/images."""
    root = Path(root)
    samples = []
    skipped_folders = []

    if not root.exists():
        print(f"  ⚠️ Path does not exist: {root}")
        return samples, skipped_folders

    for folder in sorted(root.iterdir()):
        if not folder.is_dir():
            continue

        canonical = LOWER_FOLDER_MAP.get(folder.name)
        if canonical is None:
            skipped_folders.append(folder.name)
            continue

        img_files = [
            f for f in folder.iterdir()
            if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
        ]
        for img_path in img_files:
            samples.append((str(img_path), canonical))

    return samples, skipped_folders


class TestDataset(Dataset):
    def __init__(self, samples, cls2idx, img_size):
        self.samples = samples
        self.cls2idx = cls2idx
        self.img_size = img_size
        self.transform = A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(NORM_MEAN, NORM_STD),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        p, c = self.samples[i]
        try:
            img = np.array(Image.open(p).convert("RGB"))
        except Exception:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        data = self.transform(image=img)
        return data["image"], self.cls2idx[c], p


# ============================================================
# LOAD MODEL + THRESHOLDS
# ============================================================
def load_model_and_thresholds():
    ckpt_path = MODEL_DIR / "best_siglip2_lower.pt"
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

    print(f"Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    cls2idx = ckpt["cls2idx"]
    idx2cls = {v: k for k, v in cls2idx.items()}
    classes = ckpt["classes"]

    print(f"  Classes: {classes}")
    print(f"  Best training F1: {ckpt['best_f1']:.4f}")

    model = SigLIP2Classifier(SIGLIP2_MODEL_ID, len(cls2idx), dropout_rate=0.0).to(DEVICE)
    model.vision.load_state_dict(ckpt["vision_model"])
    model.classifier.load_state_dict(ckpt["classifier"])
    model.eval()
    print(f"  ✅ Model loaded")

    thresh_path = MODEL_DIR / "lower_siglip2_thresholds.json"
    if thresh_path.exists():
        with open(thresh_path) as f:
            thresholds = json.load(f)["thresholds"]
        print(f"\n  Thresholds:")
        for c in sorted(thresholds.keys()):
            print(f"    {c:20s}: {thresholds[c]:.2f}")
    else:
        print(f"  ⚠️ No threshold file, using 0.5")
        thresholds = {c: 0.5 for c in classes}

    return model, cls2idx, idx2cls, classes, thresholds


# ============================================================
# INFERENCE
# ============================================================
@torch.no_grad()
def run_inference(model, dataloader, idx2cls):
    all_paths, all_true, all_pred_idx, all_pred_lbl, all_conf, all_probs = [], [], [], [], [], []

    for xb, yb, paths in tqdm(dataloader, desc="Inference"):
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda"):
            logits = model(xb)

        probs = F.softmax(logits, dim=1).cpu().numpy()
        pred_idx = logits.argmax(1).cpu().numpy()
        conf = probs[np.arange(len(pred_idx)), pred_idx]

        all_paths.extend(paths)
        all_true.extend(yb.numpy().tolist())
        all_pred_idx.extend(pred_idx.tolist())
        all_pred_lbl.extend([idx2cls[p] for p in pred_idx])
        all_conf.extend(conf.tolist())
        all_probs.append(probs)

    return {
        "paths": all_paths,
        "true_labels": np.array(all_true),
        "pred_indices": np.array(all_pred_idx),
        "pred_labels": all_pred_lbl,
        "confidences": np.array(all_conf),
        "probs": np.concatenate(all_probs, axis=0),
    }


# ============================================================
# EVALUATION
# ============================================================
def evaluate(results, cls2idx, idx2cls, classes, thresholds, dataset_name):
    y_true = results["true_labels"]
    y_pred = results["pred_indices"]
    confidences = results["confidences"]
    paths = results["paths"]
    probs = results["probs"]

    # ── RAW METRICS ──
    print(f"\n{'='*65}")
    print(f"  [{dataset_name}] RAW METRICS (no threshold)")
    print(f"{'='*65}")

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    print(f"\n  Accuracy:      {acc:.4f}")
    print(f"  F1 (macro):    {f1_macro:.4f}")
    print(f"  F1 (weighted): {f1_weighted:.4f}")

    rep = classification_report(y_true, y_pred, target_names=classes, digits=4)
    print(f"\n{rep}")

    cm = confusion_matrix(y_true, y_pred)
    print(f"  Confusion Matrix:")
    header = "               " + "  ".join(f"{c[:8]:>8s}" for c in classes)
    print(header)
    for i, cls_name in enumerate(classes):
        row = f"  {cls_name[:12]:>12s}  " + "  ".join(f"{cm[i,j]:>8d}" for j in range(len(classes)))
        print(row)

    # ── THRESHOLD-FILTERED ──
    print(f"\n{'='*65}")
    print(f"  [{dataset_name}] THRESHOLD-FILTERED METRICS")
    print(f"{'='*65}")

    print(f"\n  {'Class':<20s} {'Thr':>6s} {'Total':>7s} {'Accept':>7s} {'Reject':>7s} "
          f"{'Rej%':>6s} {'Acc_raw':>8s} {'Acc_filt':>9s}")
    print(f"  {'-'*80}")

    total_accepted = 0
    total_correct_accepted = 0
    total_samples = 0

    for cls_name in classes:
        cls_idx = cls2idx[cls_name]
        thr = thresholds.get(cls_name, 0.5)
        pred_mask = (y_pred == cls_idx)
        n_total = pred_mask.sum()

        if n_total == 0:
            print(f"  {cls_name:<20s} {thr:>6.2f} {0:>7d} {0:>7d} {0:>7d} {'N/A':>6s} {'N/A':>8s} {'N/A':>9s}")
            continue

        raw_correct = (y_true[pred_mask] == cls_idx).sum()
        raw_acc = raw_correct / n_total

        conf_mask = confidences[pred_mask] >= thr
        n_accepted = conf_mask.sum()
        n_rejected = n_total - n_accepted

        if n_accepted > 0:
            filt_correct = (y_true[pred_mask][conf_mask] == cls_idx).sum()
            filt_acc = filt_correct / n_accepted
            total_correct_accepted += filt_correct
        else:
            filt_acc = 0.0

        total_accepted += n_accepted
        total_samples += n_total
        rej_pct = f"{n_rejected / n_total * 100:.1f}%"
        print(f"  {cls_name:<20s} {thr:>6.2f} {n_total:>7d} {n_accepted:>7d} {n_rejected:>7d} "
              f"{rej_pct:>6s} {raw_acc:>8.3f} {filt_acc:>9.3f}")

    if total_accepted > 0:
        print(f"\n  Overall filtered accuracy: {total_correct_accepted / total_accepted:.4f}")
        print(f"  Overall rejection rate:    {(total_samples - total_accepted) / total_samples:.4f}")
        print(f"  Accepted: {total_accepted}/{total_samples}")

    # ── CONFIDENCE DISTRIBUTION ──
    print(f"\n{'='*65}")
    print(f"  [{dataset_name}] CONFIDENCE DISTRIBUTION")
    print(f"{'='*65}")

    correct_mask = (y_true == y_pred)
    wrong_mask = ~correct_mask

    if correct_mask.sum() > 0:
        print(f"\n  Correct: mean={confidences[correct_mask].mean():.3f}  "
              f"median={np.median(confidences[correct_mask]):.3f}  "
              f"min={confidences[correct_mask].min():.3f}")
    if wrong_mask.sum() > 0:
        print(f"  Wrong:   mean={confidences[wrong_mask].mean():.3f}  "
              f"median={np.median(confidences[wrong_mask]):.3f}  "
              f"max={confidences[wrong_mask].max():.3f}")
        gap = confidences[correct_mask].mean() - confidences[wrong_mask].mean()
        print(f"  Gap:     {gap:.3f}")

    # ── MISCLASSIFIED ──
    misclassified = []
    for i in range(len(y_true)):
        if y_true[i] != y_pred[i]:
            misclassified.append({
                "path": paths[i],
                "true_label": idx2cls[y_true[i]],
                "pred_label": idx2cls[y_pred[i]],
                "confidence": round(float(confidences[i]), 4),
                "true_class_prob": round(float(probs[i, y_true[i]]), 4),
            })
    misclassified.sort(key=lambda x: x["confidence"], reverse=True)

    print(f"\n{'='*65}")
    print(f"  [{dataset_name}] MISCLASSIFIED: {len(misclassified)} / {len(y_true)}")
    print(f"{'='*65}")

    print(f"\n  Top high-confidence mistakes:")
    print(f"  {'True':<16s} {'Pred':<16s} {'Conf':>6s} {'TrueP':>6s}  File")
    print(f"  {'-'*75}")
    for m in misclassified[:20]:
        fname = Path(m["path"]).name[:30]
        print(f"  {m['true_label']:<16s} {m['pred_label']:<16s} {m['confidence']:>6.3f} "
              f"{m['true_class_prob']:>6.3f}  {fname}")

    # ── ERROR PAIRS ──
    print(f"\n  Error pairs:")
    for cls_name in classes:
        wrong_from = [m for m in misclassified if m["true_label"] == cls_name]
        if not wrong_from:
            print(f"    {cls_name}: 0 errors ✅")
        else:
            pairs = Counter(m["pred_label"] for m in wrong_from)
            desc = ", ".join(f"{k}({v})" for k, v in pairs.most_common(3))
            print(f"    {cls_name}: {len(wrong_from)} errors → {desc}")

    # ── JOGGING_BOTTOMS ↔ TROUSERS DEEP DIVE ──
    jb_idx = cls2idx.get("jogging_bottoms")
    tr_idx = cls2idx.get("trousers")

    if jb_idx is not None and tr_idx is not None:
        print(f"\n{'='*65}")
        print(f"  [{dataset_name}] DEEP DIVE: jogging_bottoms ↔ trousers")
        print(f"{'='*65}")

        jb_as_tr = [m for m in misclassified
                    if m["true_label"] == "jogging_bottoms" and m["pred_label"] == "trousers"]
        tr_as_jb = [m for m in misclassified
                    if m["true_label"] == "trousers" and m["pred_label"] == "jogging_bottoms"]

        n_jb = int((y_true == jb_idx).sum())
        n_tr = int((y_true == tr_idx).sum())

        print(f"\n  jogging_bottoms → trousers: {len(jb_as_tr)}/{n_jb} ({len(jb_as_tr)/max(n_jb,1)*100:.1f}%)")
        print(f"  trousers → jogging_bottoms: {len(tr_as_jb)}/{n_tr} ({len(tr_as_jb)/max(n_tr,1)*100:.1f}%)")

        if jb_as_tr:
            confs = [m["confidence"] for m in jb_as_tr]
            print(f"  joggers→trousers conf: mean={np.mean(confs):.3f}, max={np.max(confs):.3f}")
            print(f"    Worst files:")
            for m in jb_as_tr[:5]:
                print(f"      {Path(m['path']).name[:40]}  conf={m['confidence']:.3f}")

        if tr_as_jb:
            confs = [m["confidence"] for m in tr_as_jb]
            print(f"  trousers→joggers conf: mean={np.mean(confs):.3f}, max={np.max(confs):.3f}")
            print(f"    Worst files:")
            for m in tr_as_jb[:5]:
                print(f"      {Path(m['path']).name[:40]}  conf={m['confidence']:.3f}")

    # ── SAVE ──
    tag = dataset_name.replace(" ", "_")

    with open(EVAL_DIR / f"misclassified_lower_{tag}.json", "w") as f:
        json.dump(misclassified, f, indent=2)

    eval_summary = {
        "model": "SigLIP2_SO400M_Lower",
        "dataset": dataset_name,
        "total_samples": int(len(y_true)),
        "accuracy": float(acc),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "n_misclassified": len(misclassified),
        "per_class_counts": {idx2cls[i]: int((y_true == i).sum()) for i in range(len(classes))},
    }

    with open(EVAL_DIR / f"eval_summary_lower_{tag}.json", "w") as f:
        json.dump(eval_summary, f, indent=2)

    np.savez(EVAL_DIR / f"predictions_lower_{tag}.npz",
             y_true=y_true, y_pred=y_pred, confidences=confidences, probs=probs, classes=classes)
    (EVAL_DIR / f"report_lower_{tag}.txt").write_text(rep)
    np.save(EVAL_DIR / f"cm_lower_{tag}.npy", cm)

    print(f"\n  Saved → {EVAL_DIR}/*_{tag}.*")
    return eval_summary


# ============================================================
# MAIN
# ============================================================
def main():
    print(f"{'='*65}")
    print(f"  SigLIP2 Lower — Eval on Multiple Test Datasets")
    print(f"{'='*65}")
    print(f"  Device: {DEVICE}")

    model, cls2idx, idx2cls, classes, thresholds = load_model_and_thresholds()

    all_summaries = {}

    for ds_name, ds_path in TEST_DATASETS.items():
        print(f"\n\n{'#'*65}")
        print(f"  DATASET: {ds_name}")
        print(f"  PATH:    {ds_path}")
        print(f"{'#'*65}")

        test_samples, skipped = collect_test_samples(ds_path)

        if skipped:
            print(f"\n  Skipped folders (not lower): {', '.join(skipped)}")

        if not test_samples:
            print(f"\n  ❌ No lower samples found")
            continue

        test_counts = Counter(c for _, c in test_samples)
        print(f"\n  Lower samples: {len(test_samples)}")
        for cls, cnt in test_counts.most_common():
            print(f"    {cls:20s}: {cnt}")

        test_ds = TestDataset(test_samples, cls2idx, IMG_SIZE)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True)

        results = run_inference(model, test_loader, idx2cls)
        summary = evaluate(results, cls2idx, idx2cls, classes, thresholds, ds_name)
        all_summaries[ds_name] = summary

    # ── CROSS-DATASET COMPARISON ──
    if len(all_summaries) > 1:
        print(f"\n\n{'='*65}")
        print(f"  COMPARISON ACROSS DATASETS")
        print(f"{'='*65}")
        print(f"\n  {'Dataset':<25s} {'Samples':>8s} {'Acc':>8s} {'F1_macro':>9s} {'Errors':>8s}")
        print(f"  {'-'*60}")
        for name, s in all_summaries.items():
            print(f"  {name:<25s} {s['total_samples']:>8d} {s['accuracy']:>8.4f} "
                  f"{s['f1_macro']:>9.4f} {s['n_misclassified']:>8d}")

    print(f"\n\nDONE. All results in: {EVAL_DIR}")


main()

  SigLIP2 Lower — Eval on Multiple Test Datasets
  Device: cuda
Loading checkpoint: /content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/best_siglip2_lower.pt
  Classes: ['jeans', 'jogging_bottoms', 'skirts', 'trousers']
  Best training F1: 0.9187


config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

  ✅ Model loaded

  Thresholds:
    jeans               : 0.48
    jogging_bottoms     : 0.20
    skirts              : 0.20
    trousers            : 0.20


#################################################################
  DATASET: classified_13_14
  PATH:    /content/drive/Shareddrives/Garment Type/classified_images_13_14_test
#################################################################

  Skipped folders (not lower): 13_14th_test, blazer, bra, dresses, fleece, jumpers, kincker, shirt, t-shirt, towel, unknown

  Lower samples: 592
    jogging_bottoms     : 249
    trousers            : 245
    jeans               : 60
    skirts              : 38


Inference: 100%|██████████| 25/25 [03:28<00:00,  8.33s/it]



  [classified_13_14] RAW METRICS (no threshold)

  Accuracy:      0.8885
  F1 (macro):    0.8275
  F1 (weighted): 0.8907

                 precision    recall  f1-score   support

          jeans     0.8056    0.9667    0.8788        60
jogging_bottoms     0.9289    0.8916    0.9098       249
         skirts     0.5556    0.6579    0.6024        38
       trousers     0.9364    0.9020    0.9189       245

       accuracy                         0.8885       592
      macro avg     0.8066    0.8545    0.8275       592
   weighted avg     0.8955    0.8885    0.8907       592

  Confusion Matrix:
                  jeans  jogging_    skirts  trousers
         jeans        58         0         0         2
  jogging_bott         0       222        18         9
        skirts         9         0        25         4
      trousers         5        17         2       221

  [classified_13_14] THRESHOLD-FILTERED METRICS

  Class                   Thr   Total  Accept  Reject   Rej%  Acc_raw  Acc

Inference: 100%|██████████| 6/6 [01:12<00:00, 12.02s/it]



  [careys] RAW METRICS (no threshold)

  Accuracy:      0.8504
  F1 (macro):    0.8153
  F1 (weighted): 0.8555

                 precision    recall  f1-score   support

          jeans     0.8857    0.8857    0.8857        35
jogging_bottoms     0.9048    0.6786    0.7755        28
         skirts     0.5238    1.0000    0.6875        11
       trousers     0.9400    0.8868    0.9126        53

       accuracy                         0.8504       127
      macro avg     0.8136    0.8628    0.8153       127
   weighted avg     0.8812    0.8504    0.8555       127

  Confusion Matrix:
                  jeans  jogging_    skirts  trousers
         jeans        31         0         3         1
  jogging_bott         0        19         7         2
        skirts         0         0        11         0
      trousers         4         2         0        47

  [careys] THRESHOLD-FILTERED METRICS

  Class                   Thr   Total  Accept  Reject   Rej%  Acc_raw  Acc_filt
  ------------

In [ ]:
"""
SigLIP2 Sub-Classifier — PT → ONNX Conversion & Validation
============================================================
Converts both upper and lower SigLIP2 .pt checkpoints to .onnx
with comprehensive validation before TRT conversion on Jetson.

Validations performed:
  1. Checkpoint integrity (keys, shapes, class names)
  2. PyTorch inference sanity check
  3. ONNX export with softmax baked in
  4. ONNX model structure verification
  5. ONNX Runtime numerical validation (PyTorch vs ONNX output match)
  6. Output tensor checks (softmax sum=1, correct shape, correct name)
  7. Batch of real images validation (if test images available)
  8. ONNX simplification (onnxsim)
  9. Final simplified ONNX vs PyTorch validation
  10. Production config JSON export

Run in Colab:
  Cell 1: !pip install --upgrade transformers onnx onnxruntime-gpu onnxsim --quiet
  Cell 2: Run this script
"""
!pip install onnx onnxruntime onnxscript
import os, json, sys, warnings, time
from pathlib import Path
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torch import nn

import onnx
import onnxruntime as ort

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG — UPDATE THESE PATHS
# ============================================================

MODELS = {
    "upper": {
        "pt_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_upper_sub/best_siglip2_upper.pt",
        "onnx_dir": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_upper_sub",
        "onnx_name": "siglip2_upper",
        "thresh_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_upper_sub/upper_siglip2_thresholds.json",
        # Optional: path to a folder with a few test images for real-image validation
        "test_images_dir": None,  # e.g., "/content/drive/.../classified_images_13_14_test/shirt"
    },
    "lower": {
        "pt_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/best_siglip2_lower.pt",
        "onnx_dir": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub",
        "onnx_name": "siglip2_lower",
        "thresh_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/lower_siglip2_thresholds.json",
        "test_images_dir": None,
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
HIDDEN_SIZE = 1152
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)

# Tolerance for numerical comparison
ATOL = 1e-4   # absolute tolerance
RTOL = 1e-3   # relative tolerance


# ============================================================
# MODEL DEFINITIONS
# ============================================================
class SigLIP2Classifier(nn.Module):
    """Must match training definition exactly."""
    def __init__(self, model_id, num_classes, dropout_rate=0.0):
        super().__init__()
        from transformers import AutoModel
        full_model = AutoModel.from_pretrained(model_id, ignore_mismatched_sizes=True)
        self.vision = full_model.vision_model
        del full_model

        self.classifier = nn.Sequential(
            nn.LayerNorm(HIDDEN_SIZE),
            nn.Dropout(dropout_rate),
            nn.Linear(HIDDEN_SIZE, num_classes),
        )

    def forward(self, pixel_values):
        outputs = self.vision(pixel_values=pixel_values)
        pooled = outputs.pooler_output
        return self.classifier(pooled)


class SigLIP2ForExport(nn.Module):
    """Wrapper: bakes softmax into output for TRT deployment.
    Output tensor is 'probabilities' — do NOT apply softmax again."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        logits = self.model(pixel_values)
        return F.softmax(logits, dim=1)


# ============================================================
# VALIDATION HELPERS
# ============================================================
def passed(msg):
    print(f"  ✅ {msg}")

def failed(msg):
    print(f"  ❌ {msg}")
    return False

def warn(msg):
    print(f"  ⚠️  {msg}")


def preprocess_image(img_path):
    """Preprocess a single image matching production pipeline."""
    img = Image.open(img_path).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - np.array(NORM_MEAN)) / np.array(NORM_STD)
    arr = arr.transpose(2, 0, 1)  # HWC → CHW
    return arr[np.newaxis, ...]   # add batch dim


# ============================================================
# MAIN CONVERSION + VALIDATION
# ============================================================
def convert_and_validate(branch, config):
    print(f"\n{'#'*70}")
    print(f"  CONVERTING: SigLIP2 {branch.upper()} Sub-Classifier")
    print(f"{'#'*70}")

    pt_path = Path(config["pt_path"])
    onnx_dir = Path(config["onnx_dir"])
    onnx_name = config["onnx_name"]
    all_passed = True

    # ──────────────────────────────────────────────
    # 1. CHECKPOINT INTEGRITY
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [1/10] Checkpoint Integrity")
    print(f"{'─'*50}")

    if not pt_path.exists():
        return failed(f"Checkpoint not found: {pt_path}")

    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)

    required_keys = {"vision_model", "classifier", "classes", "cls2idx", "backbone",
                     "hidden_size", "img_size", "best_f1", "normalization"}
    missing = required_keys - set(ckpt.keys())
    if missing:
        all_passed = failed(f"Missing checkpoint keys: {missing}")
    else:
        passed("All required keys present")

    classes = ckpt["classes"]
    cls2idx = ckpt["cls2idx"]
    num_classes = len(classes)
    print(f"    Classes ({num_classes}): {classes}")
    print(f"    Best F1: {ckpt['best_f1']:.4f}")
    print(f"    Backbone: {ckpt['backbone']}")
    print(f"    img_size: {ckpt['img_size']}")
    print(f"    hidden_size: {ckpt['hidden_size']}")

    if ckpt["img_size"] != IMG_SIZE:
        all_passed = failed(f"img_size mismatch: ckpt={ckpt['img_size']} vs expected={IMG_SIZE}")
    else:
        passed(f"img_size matches: {IMG_SIZE}")

    if ckpt["hidden_size"] != HIDDEN_SIZE:
        all_passed = failed(f"hidden_size mismatch: ckpt={ckpt['hidden_size']} vs expected={HIDDEN_SIZE}")
    else:
        passed(f"hidden_size matches: {HIDDEN_SIZE}")

    # Check classifier weight shape
    clf_weight_key = "2.weight"  # Sequential index: LayerNorm(0), Dropout(1), Linear(2)
    if clf_weight_key in ckpt["classifier"]:
        shape = ckpt["classifier"][clf_weight_key].shape
        expected = (num_classes, HIDDEN_SIZE)
        if shape != expected:
            all_passed = failed(f"Classifier weight shape {shape} != expected {expected}")
        else:
            passed(f"Classifier weight shape: {shape}")
    else:
        warn("Could not find classifier linear weight for shape check")

    norm = ckpt.get("normalization", {})
    print(f"    Normalization: mean={norm.get('mean')}, std={norm.get('std')}")

    # ──────────────────────────────────────────────
    # 2. LOAD MODEL + PYTORCH SANITY CHECK
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [2/10] PyTorch Model Loading & Sanity Check")
    print(f"{'─'*50}")

    model = SigLIP2Classifier(SIGLIP2_MODEL_ID, num_classes, dropout_rate=0.0).to(DEVICE)
    model.vision.load_state_dict(ckpt["vision_model"])
    model.classifier.load_state_dict(ckpt["classifier"])
    model.eval()
    passed("Model loaded successfully")

    # Random input sanity check
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    with torch.no_grad():
        logits = model(dummy)

    if logits.shape != (1, num_classes):
        all_passed = failed(f"Logits shape {logits.shape} != expected (1, {num_classes})")
    else:
        passed(f"Logits shape: {logits.shape}")

    probs_pt = F.softmax(logits, dim=1)
    prob_sum = probs_pt.sum().item()
    if abs(prob_sum - 1.0) > 1e-4:
        all_passed = failed(f"Softmax sum = {prob_sum:.6f} (expected 1.0)")
    else:
        passed(f"Softmax sum: {prob_sum:.6f}")

    print(f"    Sample probs: {probs_pt.cpu().numpy().round(4)}")

    # ──────────────────────────────────────────────
    # 3. ONNX EXPORT
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [3/10] ONNX Export")
    print(f"{'─'*50}")

    export_model = SigLIP2ForExport(model).eval().to(DEVICE)

    # Verify export wrapper
    with torch.no_grad():
        export_out = export_model(dummy)
    if not torch.allclose(export_out, probs_pt, atol=1e-6):
        all_passed = failed("Export wrapper output differs from manual softmax!")
    else:
        passed("Export wrapper matches manual softmax")

    onnx_path = onnx_dir / f"{onnx_name}.onnx"
    print(f"    Exporting to: {onnx_path}")

    t0 = time.time()
    with torch.no_grad():
        torch.onnx.export(
            export_model,
            dummy,
            str(onnx_path),
            opset_version=18,
            input_names=["pixel_values"],
            output_names=["probabilities"],
            dynamic_axes=None,  # fixed batch=1 for TRT
        )
    export_time = time.time() - t0

    if not onnx_path.exists():
        return failed("ONNX file not created!")

    file_size_mb = os.path.getsize(onnx_path) / 1e6
    passed(f"ONNX exported: {file_size_mb:.0f} MB in {export_time:.1f}s")

    # ──────────────────────────────────────────────
    # 4. ONNX STRUCTURE VERIFICATION
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [4/10] ONNX Structure Verification")
    print(f"{'─'*50}")

    onnx_model = onnx.load(str(onnx_path))

    try:
        onnx.checker.check_model(onnx_model)
        passed("ONNX checker passed")
    except Exception as e:
        all_passed = failed(f"ONNX checker failed: {e}")

    # Check input
    inp = onnx_model.graph.input[0]
    inp_name = inp.name
    inp_shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"    Input:  name='{inp_name}', shape={inp_shape}")

    if inp_name != "pixel_values":
        all_passed = failed(f"Input name '{inp_name}' != expected 'pixel_values'")
    else:
        passed("Input name: pixel_values")

    if inp_shape != [1, 3, IMG_SIZE, IMG_SIZE]:
        all_passed = failed(f"Input shape {inp_shape} != expected [1, 3, {IMG_SIZE}, {IMG_SIZE}]")
    else:
        passed(f"Input shape: {inp_shape}")

    # Check output
    out = onnx_model.graph.output[0]
    out_name = out.name
    out_shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"    Output: name='{out_name}', shape={out_shape}")

    if out_name != "probabilities":
        all_passed = failed(f"Output name '{out_name}' != expected 'probabilities'")
    else:
        passed("Output name: probabilities")

    if out_shape != [1, num_classes]:
        all_passed = failed(f"Output shape {out_shape} != expected [1, {num_classes}]")
    else:
        passed(f"Output shape: [1, {num_classes}]")

    del onnx_model  # free memory

    # ──────────────────────────────────────────────
    # 5. ONNX RUNTIME NUMERICAL VALIDATION
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [5/10] ONNX Runtime Numerical Validation")
    print(f"{'─'*50}")

    # Use CPU provider for validation (works on all setups)
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    try:
        sess = ort.InferenceSession(str(onnx_path), providers=providers)
        active_provider = sess.get_providers()[0]
        passed(f"ORT session created ({active_provider})")
    except Exception as e:
        all_passed = failed(f"ORT session failed: {e}")
        sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
        passed("Fallback to CPU provider")

    # Test with multiple random inputs
    n_test = 5
    max_diff_all = 0.0
    print(f"    Testing {n_test} random inputs...")

    for i in range(n_test):
        test_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

        # PyTorch
        with torch.no_grad():
            pt_out = export_model(test_input).cpu().numpy()

        # ONNX
        ort_out = sess.run(None, {"pixel_values": test_input.cpu().numpy()})[0]

        max_diff = np.max(np.abs(pt_out - ort_out))
        max_diff_all = max(max_diff_all, max_diff)

        # Check probabilities sum to 1
        ort_sum = ort_out.sum()
        if abs(ort_sum - 1.0) > 1e-3:
            all_passed = failed(f"  Test {i+1}: ORT prob sum = {ort_sum:.6f}")

        if not np.allclose(pt_out, ort_out, atol=ATOL, rtol=RTOL):
            all_passed = failed(f"  Test {i+1}: max_diff={max_diff:.6f} exceeds tolerance")

    if max_diff_all < ATOL:
        passed(f"All {n_test} tests passed — max diff: {max_diff_all:.8f}")
    else:
        warn(f"Max diff across tests: {max_diff_all:.8f} (tolerance: atol={ATOL})")

    # ──────────────────────────────────────────────
    # 6. OUTPUT TENSOR DEEP CHECK
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [6/10] Output Tensor Deep Check")
    print(f"{'─'*50}")

    # Use a deterministic input (all 0.5 = gray image in SigLIP2 normalization)
    gray_input = np.full((1, 3, IMG_SIZE, IMG_SIZE), 0.0, dtype=np.float32)  # normalized gray
    ort_gray = sess.run(None, {"pixel_values": gray_input})[0]

    # All probabilities should be positive
    if np.all(ort_gray >= 0):
        passed("All probabilities >= 0")
    else:
        all_passed = failed(f"Negative probabilities found: min={ort_gray.min():.6f}")

    # All probabilities should be <= 1
    if np.all(ort_gray <= 1.0 + 1e-6):
        passed("All probabilities <= 1.0")
    else:
        all_passed = failed(f"Probability > 1.0 found: max={ort_gray.max():.6f}")

    # Sum should be ~1.0
    gray_sum = ort_gray.sum()
    if abs(gray_sum - 1.0) < 1e-3:
        passed(f"Probability sum: {gray_sum:.6f}")
    else:
        all_passed = failed(f"Probability sum: {gray_sum:.6f} (expected 1.0)")

    # No NaN or Inf
    if not np.any(np.isnan(ort_gray)) and not np.any(np.isinf(ort_gray)):
        passed("No NaN/Inf values")
    else:
        all_passed = failed("NaN or Inf detected in output!")

    # Check no single class dominates suspiciously (all same value = double-softmax bug)
    ort_max = ort_gray.max()
    ort_min = ort_gray.min()
    ort_range = ort_max - ort_min
    print(f"    Gray image probs: min={ort_min:.4f}, max={ort_max:.4f}, range={ort_range:.4f}")
    print(f"    Per-class: {dict(zip(classes, ort_gray[0].round(4)))}")

    if ort_range < 0.01:
        warn("Very uniform output on gray image — possible double-softmax issue")
        warn("But this is on a blank input, so may be expected")

    # ──────────────────────────────────────────────
    # 7. REAL IMAGE VALIDATION (if available)
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [7/10] Real Image Validation")
    print(f"{'─'*50}")

    test_dir = config.get("test_images_dir")
    if test_dir and Path(test_dir).exists():
        img_files = [f for f in Path(test_dir).iterdir()
                     if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}][:5]

        if img_files:
            print(f"    Testing {len(img_files)} real images from {test_dir}")
            for img_path in img_files:
                inp_np = preprocess_image(str(img_path))

                # PyTorch
                inp_pt = torch.tensor(inp_np, dtype=torch.float32, device=DEVICE)
                with torch.no_grad():
                    pt_probs = export_model(inp_pt).cpu().numpy()

                # ONNX
                ort_probs = sess.run(None, {"pixel_values": inp_np.astype(np.float32)})[0]

                max_diff = np.max(np.abs(pt_probs - ort_probs))
                pt_pred = classes[pt_probs.argmax()]
                ort_pred = classes[ort_probs.argmax()]
                ort_conf = ort_probs.max()

                match = "✅" if pt_pred == ort_pred else "❌"
                print(f"    {match} {img_path.name[:35]:35s}  PT={pt_pred:15s}  ORT={ort_pred:15s}  "
                      f"conf={ort_conf:.3f}  diff={max_diff:.6f}")

                if pt_pred != ort_pred:
                    all_passed = failed(f"Prediction mismatch on {img_path.name}")
            passed("Real image validation complete")
        else:
            warn("No images found in test directory")
    else:
        warn("No test_images_dir set — skipping real image validation")

    # ──────────────────────────────────────────────
    # 8. ONNX SIMPLIFICATION
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [8/10] ONNX Simplification (onnxsim)")
    print(f"{'─'*50}")

    sim_path = onnx_dir / f"{onnx_name}_sim.onnx"

    try:
        import onnxsim
        onnx_model = onnx.load(str(onnx_path))
        sim_model, check = onnxsim.simplify(onnx_model)

        if check:
            onnx.save(sim_model, str(sim_path))
            sim_size = os.path.getsize(sim_path) / 1e6
            orig_size = os.path.getsize(onnx_path) / 1e6
            reduction = (1 - sim_size / orig_size) * 100
            passed(f"Simplified: {orig_size:.0f}MB → {sim_size:.0f}MB ({reduction:.1f}% reduction)")
        else:
            all_passed = failed("onnxsim simplification check failed")
            sim_path = onnx_path  # fallback to original

        del onnx_model, sim_model
    except ImportError:
        warn("onnxsim not installed — skipping simplification")
        warn("Install with: pip install onnxsim")
        sim_path = onnx_path
    except Exception as e:
        warn(f"Simplification failed: {e}")
        sim_path = onnx_path

    # ──────────────────────────────────────────────
    # 9. SIMPLIFIED ONNX VALIDATION
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [9/10] Simplified ONNX Validation")
    print(f"{'─'*50}")

    if sim_path.exists() and sim_path != onnx_path:
        # Validate simplified model
        sim_onnx = onnx.load(str(sim_path))
        try:
            onnx.checker.check_model(sim_onnx)
            passed("Simplified ONNX checker passed")
        except Exception as e:
            all_passed = failed(f"Simplified ONNX checker failed: {e}")

        # Check I/O names preserved
        sim_inp_name = sim_onnx.graph.input[0].name
        sim_out_name = sim_onnx.graph.output[0].name
        if sim_inp_name == "pixel_values" and sim_out_name == "probabilities":
            passed("I/O names preserved after simplification")
        else:
            all_passed = failed(f"I/O names changed: in='{sim_inp_name}', out='{sim_out_name}'")

        del sim_onnx

        # Numerical check on simplified model
        sess_sim = ort.InferenceSession(str(sim_path), providers=providers)
        test_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

        with torch.no_grad():
            pt_out = export_model(test_input).cpu().numpy()
        sim_out = sess_sim.run(None, {"pixel_values": test_input.cpu().numpy()})[0]

        max_diff = np.max(np.abs(pt_out - sim_out))
        if max_diff < ATOL:
            passed(f"Simplified model matches PyTorch (max_diff={max_diff:.8f})")
        else:
            all_passed = failed(f"Simplified model differs: max_diff={max_diff:.6f}")

        del sess_sim
    else:
        warn("No simplified model to validate")

    # ──────────────────────────────────────────────
    # 10. PRODUCTION CONFIG EXPORT
    # ──────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  [10/10] Production Config Export")
    print(f"{'─'*50}")

    # Load thresholds
    thresh_path = config.get("thresh_path")
    thresholds = {}
    if thresh_path and Path(thresh_path).exists():
        with open(thresh_path) as f:
            thresh_data = json.load(f)
        thresholds = thresh_data.get("thresholds", {})
        passed(f"Loaded {len(thresholds)} thresholds")
    else:
        warn("No threshold file — using 0.5 defaults")
        thresholds = {c: 0.5 for c in classes}

    prod_config = {
        "model": f"SigLIP2_SO400M_{branch.title()}",
        "model_id": SIGLIP2_MODEL_ID,
        "branch": branch,
        "img_size": IMG_SIZE,
        "num_classes": num_classes,
        "class_names": classes,
        "class_to_idx": cls2idx,
        "thresholds": thresholds,
        "normalization": {
            "mean": list(NORM_MEAN),
            "std": list(NORM_STD),
        },
        "preprocessing": {
            "normalize_mean": list(NORM_MEAN),
            "normalize_std": list(NORM_STD),
            "resize": IMG_SIZE,
            "interpolation": "bilinear",
        },
        "onnx": {
            "file": sim_path.name if sim_path.exists() else onnx_path.name,
            "input_name": "pixel_values",
            "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
            "output_name": "probabilities",
            "output_shape": [1, num_classes],
            "output_includes_softmax": True,
        },
        "training_info": {
            "best_f1": float(ckpt["best_f1"]),
            "backbone": ckpt["backbone"],
            "hidden_size": HIDDEN_SIZE,
            "epoch": ckpt.get("epoch", "N/A"),
        },
        "trt_conversion": {
            "fp16_command": f"trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp16.engine --fp16 --memPoolSize=workspace:8192M",
            "fp32_command": f"trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp32.engine --memPoolSize=workspace:8192M",
            "note": "Use FP16 first. If attention layers lose precision, fall back to FP32.",
        },
        "production_notes": [
            "Output 'probabilities' already has softmax applied",
            "Do NOT apply softmax again — causes double-softmax bug (all outputs ~0.475)",
            "For temperature scaling: logits = log(clip(probs, 1e-7, 1.0)); logits /= T; probs = softmax(logits)",
        ],
    }

    config_path = onnx_dir / f"{onnx_name}_production_config.json"
    with open(config_path, "w") as f:
        json.dump(prod_config, f, indent=2)
    passed(f"Config saved: {config_path}")

    # ──────────────────────────────────────────────
    # SUMMARY
    # ──────────────────────────────────────────────
    print(f"\n{'='*70}")
    if all_passed:
        print(f"  ✅ ALL VALIDATIONS PASSED — {branch.upper()} model ready for TRT conversion")
    else:
        print(f"  ⚠️  SOME VALIDATIONS FAILED — review above before deploying")
    print(f"{'='*70}")

    print(f"\n  Files created:")
    print(f"    ONNX (original):    {onnx_path}")
    if sim_path != onnx_path:
        print(f"    ONNX (simplified):  {sim_path}  ← USE THIS FOR TRT")
    print(f"    Production config:  {config_path}")

    print(f"\n  TRT conversion commands (run on Jetson AGX Orin):")
    print(f"    # Try FP16 first (faster inference):")
    print(f"    trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp16.engine --fp16 --memPoolSize=workspace:8192M")
    print(f"\n    # If FP16 has precision issues, use FP32:")
    print(f"    trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp32.engine --memPoolSize=workspace:8192M")

    # Cleanup
    del model, export_model, sess
    torch.cuda.empty_cache()

    return all_passed


# ============================================================
# RUN BOTH
# ============================================================
print(f"{'='*70}")
print(f"  SigLIP2 Sub-Classifier — PT → ONNX Conversion Pipeline")
print(f"  Device: {DEVICE}")
print(f"{'='*70}")

results = {}
for branch, config in MODELS.items():
    results[branch] = convert_and_validate(branch, config)

# Final summary
print(f"\n\n{'='*70}")
print(f"  FINAL SUMMARY")
print(f"{'='*70}")
for branch, passed_all in results.items():
    status = "✅ READY" if passed_all else "⚠️  NEEDS REVIEW"
    print(f"  {branch.upper():>8s}: {status}")

print(f"\n  Next steps:")
print(f"    1. Copy *_sim.onnx + *_production_config.json to Jetson AGX Orin")
print(f"    2. Run trtexec to build .engine files")
print(f"    3. Update inf_macro.py to load new SigLIP2 sub-classifier engines")
print(f"    4. Production code must use normalization mean/std = [0.5, 0.5, 0.5]")
print(f"       (different from ConvNeXt's ImageNet normalization!)")
print(f"{'='*70}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.3/159.3 kB 18.7 MB/s eta 0:00:00
  SigLIP2 Sub-Classifier — PT → ONNX Conversion Pipeline
  Device: cuda

######################################################################
  CONVERTING: SigLIP2 UPPER Sub-Classifier
######################################################################

──────────────────────────────────────────────────
  [1/10] Checkpoint Integrity
──────────────────────────────────────────────────
  ✅ All required keys present
    Classes (6): ['blazer', 'dresses', 'fleece', 'jumpers', 'shirt', 't-shirt']
    Best F1: 0.9072
    Backbone: google/siglip2-so400m-patch14-384
    img_size: 384
    hidden_size: 1152
  ✅ img_size matches: 384
  ✅ hidden_size matches: 1152
  ✅ Classifie

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

  ✅ Model loaded successfully
  ✅ Logits shape: torch.Size([1, 6])
  ✅ Softmax sum: 1.000000
    Sample probs: [[0.0791 0.5001 0.1497 0.0906 0.0965 0.084 ]]

──────────────────────────────────────────────────
  [3/10] ONNX Export
──────────────────────────────────────────────────
  ✅ Export wrapper matches manual softmax
    Exporting to: /content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_upper_sub/siglip2_upper.onnx


W0221 12:04:25.517000 3095 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0221 12:04:25.519000 3095 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0221 12:04:25.521000 3095 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0221 12:04:25.522000 3095 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `SigLIP2ForExport([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SigLIP2ForExport([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


KeyboardInterrupt: 

In [ ]:
import os, json, sys, warnings, time
from pathlib import Path
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torch import nn

import onnx
# onnxruntime and onnxsim are typically installed in the notebook where the original code was run
# import onnxruntime as ort
# import onnxsim

warnings.filterwarnings("ignore")

### Configuration
Update the `MODELS` dictionary with the correct paths for your specific model (e.g., 'upper' or 'lower' sub-classifier).

In [ ]:
MODELS = {
    "lower": {
        "pt_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/best_siglip2_lower.pt",
        "onnx_dir": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub",
        "onnx_name": "siglip2_lower",
        "thresh_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/lower_siglip2_thresholds.json",
        "test_images_dir": None,
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
HIDDEN_SIZE = 1152
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)

### Model Definitions
These classes are essential for loading the PyTorch checkpoint and for wrapping the model for ONNX export with softmax included.

In [ ]:
class SigLIP2Classifier(nn.Module):
    """Must match training definition exactly."""
    def __init__(self, model_id, num_classes, dropout_rate=0.0):
        super().__init__()
        from transformers import AutoModel
        full_model = AutoModel.from_pretrained(model_id, ignore_mismatched_sizes=True)
        self.vision = full_model.vision_model
        del full_model

        self.classifier = nn.Sequential(
            nn.LayerNorm(HIDDEN_SIZE),
            nn.Dropout(dropout_rate),
            nn.Linear(HIDDEN_SIZE, num_classes),
        )

    def forward(self, pixel_values):
        # For ONNX export, we need to provide all expected inputs to the vision model.
        # Based on SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
        # Patch size is 14, Image size is 384. 384 // 14 = 27.
        # This suggests a patch grid of 27x27 = 729 patches.
        batch_size = pixel_values.shape[0]
        num_patches_per_dim = IMG_SIZE // 14 # Assuming 14 is the patch size
        num_patches_flat = num_patches_per_dim * num_patches_per_dim # 27 * 27 = 729

        # Create dummy pixel_attention_mask (all ones for full image)
        pixel_attention_mask = torch.ones(
            (batch_size, num_patches_flat),
            dtype=torch.long, device=pixel_values.device
        )

        # Create dummy spatial_shapes
        spatial_shapes = torch.tensor(
            [[num_patches_per_dim, num_patches_per_dim]],
            dtype=torch.long, device=pixel_values.device
        )

        outputs = self.vision(
            pixel_values=pixel_values,
            pixel_attention_mask=pixel_attention_mask,
            spatial_shapes=spatial_shapes
        )
        pooled = outputs.pooler_output
        return self.classifier(pooled)


class SigLIP2ForExport(nn.Module):
    """Wrapper: bakes softmax into output for TRT deployment.
    Output tensor is 'probabilities' — do NOT apply softmax again."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        logits = self.model(pixel_values)
        return F.softmax(logits, dim=1)

### ONNX Conversion Function
This function encapsulates the steps to load the PyTorch model, export it to ONNX, optionally simplify it, and save the final production configuration.

In [ ]:
def convert_and_export_model(branch, config):
    print(f"\n{'#'*70}")
    print(f"  CONVERTING: SigLIP2 {branch.upper()} Sub-Classifier")
    print(f"{'#'*70}")

    pt_path = Path(config["pt_path"])
    onnx_dir = Path(config["onnx_dir"])
    onnx_name = config["onnx_name"]
    onnx_dir.mkdir(parents=True, exist_ok=True)

    if not pt_path.exists():
        print(f"❌ Checkpoint not found: {pt_path}")
        return False

    print(f"Loading PyTorch checkpoint from: {pt_path}")
    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)

    classes = ckpt["classes"]
    cls2idx = ckpt["cls2idx"]
    num_classes = len(classes)
    print(f"  Classes ({num_classes}): {classes}")

    print(f"Loading SigLIP2Classifier model...")
    model = SigLIP2Classifier(SIGLIP2_MODEL_ID, num_classes, dropout_rate=0.0).to(DEVICE)
    model.vision.load_state_dict(ckpt["vision_model"])
    model.classifier.load_state_dict(ckpt["classifier"])
    model.eval()
    print("  ✅ PyTorch model loaded successfully")

    export_model = SigLIP2ForExport(model).eval().to(DEVICE)

    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    onnx_path = onnx_dir / f"{onnx_name}.onnx"

    print(f"\nExporting ONNX model to: {onnx_path}")
    with torch.no_grad():
        torch.onnx.export(
            export_model,
            dummy_input,
            str(onnx_path),
            opset_version=18,
            input_names=["pixel_values"],
            output_names=["probabilities"],
            dynamic_axes=None,
        )
    print(f"  ✅ ONNX model exported. File size: {os.path.getsize(onnx_path) / 1e6:.0f} MB")

    # Verify ONNX model
    try:
        onnx_model = onnx.load(str(onnx_path))
        onnx.checker.check_model(onnx_model)
        print("  ✅ ONNX model verification passed (onnx.checker)")
    except Exception as e:
        print(f"  ❌ ONNX model verification failed: {e}")
        return False

    sim_path = onnx_path # Default to original if simplification fails or not available
    try:
        import onnxsim
        print("\nAttempting ONNX simplification...")
        sim_model, check = onnxsim.simplify(onnx.load(str(onnx_path)))
        if check:
            sim_path = onnx_dir / f"{onnx_name}_sim.onnx"
            onnx.save(sim_model, str(sim_path))
            orig_size = os.path.getsize(onnx_path) / 1e6
            sim_size = os.path.getsize(sim_path) / 1e6
            reduction = (1 - sim_size / orig_size) * 100
            print(f"  ✅ ONNX model simplified: {orig_size:.0f}MB -> {sim_size:.0f}MB ({reduction:.1f}% reduction)")
        else:
            print("  ⚠️  ONNX simplification check failed.")
    except ImportError:
        print("  ⚠️  onnxsim not installed. Skipping simplification. Install with: `pip install onnxsim`")
    except Exception as e:
        print(f"  ❌ ONNX simplification failed: {e}")

    # Save production config
    thresh_path = config.get("thresh_path")
    thresholds = {}
    if thresh_path and Path(thresh_path).exists():
        with open(thresh_path) as f:
            thresh_data = json.load(f)
        thresholds = thresh_data.get("thresholds", {})
        print(f"  Loaded {len(thresholds)} thresholds from {thresh_path}")
    else:
        print("  ⚠️  No threshold file found, using default 0.5 for all classes.")
        thresholds = {c: 0.5 for c in classes}

    prod_config = {
        "model": f"SigLIP2_SO400M_{branch.title()}",
        "model_id": SIGLIP2_MODEL_ID,
        "branch": branch,
        "img_size": IMG_SIZE,
        "num_classes": num_classes,
        "class_names": classes,
        "class_to_idx": cls2idx,
        "thresholds": thresholds,
        "normalization": {
            "mean": list(NORM_MEAN),
            "std": list(NORM_STD),
        },
        "onnx": {
            "file": sim_path.name,
            "input_name": "pixel_values",
            "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
            "output_name": "probabilities",
            "output_shape": [1, num_classes],
            "output_includes_softmax": True,
        },
        "training_info": {
            "best_f1": float(ckpt["best_f1"]),
            "backbone": ckpt["backbone"],
            "hidden_size": HIDDEN_SIZE,
            "epoch": ckpt.get("epoch", "N/A"),
        },
        "trt_conversion": {
            "fp16_command": f"trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp16.engine --fp16 --memPoolSize=workspace:8192M",
            "fp32_command": f"trtexec --onnx={sim_path.name} --saveEngine={onnx_name}_fp32.engine --memPoolSize=workspace:8192M",
            "note": "Use FP16 first. If attention layers lose precision, fall back to FP32.",
        },
        "production_notes": [
            "Output 'probabilities' already has softmax applied",
            "Do NOT apply softmax again — causes double-softmax bug (all outputs ~0.475)",
        ],
    }

    config_path = onnx_dir / f"{onnx_name}_production_config.json"
    with open(config_path, "w") as f:
        json.dump(prod_config, f, indent=2)
    print(f"  ✅ Production config saved: {config_path}")
    print(f"\n  ONNX model for TRT: {sim_path}")
    print(f"  TRT FP16 Command: {prod_config['trt_conversion']['fp16_command']}")
    print(f"  TRT FP32 Command: {prod_config['trt_conversion']['fp32_command']}")
    return True

### Run Conversion
Execute this cell to perform the conversion for the configured models.

In [ ]:
for branch, config in MODELS.items():
    convert_and_export_model(branch, config)

print("\nConversion process completed.")

In [ ]:
import os
import json
import sys
import warnings
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

import onnx

warnings.filterwarnings("ignore")


MODELS = {
    "lower": {
        "pt_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/best_siglip2_lower.pt",
        "onnx_dir": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub",
        "onnx_name": "siglip2_lower",
        "thresh_path": "/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/lower_siglip2_thresholds.json",
        "test_images_dir": None,
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
PATCH_SIZE = 14
HIDDEN_SIZE = 1152
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)
OPSET_VERSION = 18
VERIFY_BATCHES = [1, 4]


class SigLIP2Classifier(nn.Module):
    """Must match training definition exactly."""
    def __init__(self, model_id, num_classes, dropout_rate=0.0):
        super().__init__()
        from transformers import AutoModel

        full_model = AutoModel.from_pretrained(
            model_id,
            ignore_mismatched_sizes=True
        )
        self.vision = full_model.vision_model
        del full_model

        self.classifier = nn.Sequential(
            nn.LayerNorm(HIDDEN_SIZE),
            nn.Dropout(dropout_rate),
            nn.Linear(HIDDEN_SIZE, num_classes),
        )

    def forward(self, pixel_values):
        # Dynamic batch-safe auxiliary inputs
        batch_size = pixel_values.shape[0]
        num_patches_per_dim = IMG_SIZE // PATCH_SIZE
        num_patches_flat = num_patches_per_dim * num_patches_per_dim

        # [B, num_patches]
        pixel_attention_mask = torch.ones(
            (batch_size, num_patches_flat),
            dtype=torch.long,
            device=pixel_values.device
        )

        # IMPORTANT: make this batch-aware too, not [1, 2]
        # [B, 2], each row is [H_patches, W_patches]
        spatial_shapes = torch.tensor(
            [num_patches_per_dim, num_patches_per_dim],
            dtype=torch.long,
            device=pixel_values.device
        ).unsqueeze(0).repeat(batch_size, 1)

        outputs = self.vision(
            pixel_values=pixel_values,
            pixel_attention_mask=pixel_attention_mask,
            spatial_shapes=spatial_shapes
        )
        pooled = outputs.pooler_output
        return self.classifier(pooled)


class SigLIP2ForExport(nn.Module):
    """
    Wrapper: bakes softmax into output for TRT deployment.
    Output tensor is 'probabilities' — do NOT apply softmax again.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        logits = self.model(pixel_values)
        return F.softmax(logits, dim=1)


def get_tensor_shape_proto(value_info):
    dims = []
    for d in value_info.type.tensor_type.shape.dim:
        if d.dim_param:
            dims.append(d.dim_param)
        elif d.dim_value:
            dims.append(d.dim_value)
        else:
            dims.append("?")
    return dims


def print_onnx_shapes(onnx_path: Path):
    print("\n--- ONNX Graph I/O Shapes ---")
    model = onnx.load(str(onnx_path))
    for x in model.graph.input:
        print(f"INPUT  {x.name}: {get_tensor_shape_proto(x)}")
    for x in model.graph.output:
        print(f"OUTPUT {x.name}: {get_tensor_shape_proto(x)}")


def verify_onnxruntime(onnx_path: Path, export_model: nn.Module):
    try:
        import onnxruntime as ort
    except ImportError:
        print("  ⚠️ onnxruntime not installed. Skipping runtime verification.")
        return

    print("\n--- Verifying ONNX Runtime output shapes ---")
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])

    for b in VERIFY_BATCHES:
        x = torch.randn(b, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

        with torch.no_grad():
            pt_out = export_model(x).detach().cpu().numpy()

        ort_out = sess.run(
            ["probabilities"],
            {"pixel_values": x.detach().cpu().numpy()}
        )[0]

        max_abs_diff = np.max(np.abs(pt_out - ort_out))
        print(
            f"  Batch {b}: "
            f"PyTorch={pt_out.shape}, ONNX={ort_out.shape}, "
            f"max_abs_diff={max_abs_diff:.8f}"
        )


def simplify_onnx(onnx_path: Path, onnx_dir: Path, onnx_name: str):
    sim_path = onnx_path
    try:
        import onnxsim
        print("\nAttempting ONNX simplification...")

        model = onnx.load(str(onnx_path))
        sim_model, check = onnxsim.simplify(model)

        if check:
            sim_path = onnx_dir / f"{onnx_name}_sim.onnx"
            onnx.save(sim_model, str(sim_path))
            orig_size = os.path.getsize(onnx_path) / 1e6
            sim_size = os.path.getsize(sim_path) / 1e6
            reduction = (1 - sim_size / orig_size) * 100 if orig_size > 0 else 0
            print(
                f"  ✅ ONNX model simplified: "
                f"{orig_size:.0f}MB -> {sim_size:.0f}MB ({reduction:.1f}% reduction)"
            )
        else:
            print("  ⚠️ ONNX simplification check failed, using original ONNX.")

    except ImportError:
        print("  ⚠️ onnxsim not installed. Skipping simplification.")
    except Exception as e:
        print(f"  ❌ ONNX simplification failed: {e}")

    return sim_path


def safe_float(value, default=None):
    try:
        return float(value)
    except Exception:
        return default


def convert_and_export_model(branch, config):
    print(f"\n{'#' * 70}")
    print(f"  CONVERTING: SigLIP2 {branch.upper()} Sub-Classifier")
    print(f"{'#' * 70}")

    pt_path = Path(config["pt_path"])
    onnx_dir = Path(config["onnx_dir"])
    onnx_name = config["onnx_name"]
    onnx_dir.mkdir(parents=True, exist_ok=True)

    if not pt_path.exists():
        print(f"❌ Checkpoint not found: {pt_path}")
        return False

    print(f"Loading PyTorch checkpoint from: {pt_path}")
    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)

    classes = ckpt["classes"]
    cls2idx = ckpt["cls2idx"]
    num_classes = len(classes)
    print(f"  Classes ({num_classes}): {classes}")

    print("Loading SigLIP2Classifier model...")
    model = SigLIP2Classifier(SIGLIP2_MODEL_ID, num_classes, dropout_rate=0.0).to(DEVICE)
    model.vision.load_state_dict(ckpt["vision_model"])
    model.classifier.load_state_dict(ckpt["classifier"])
    model.eval()
    print("  ✅ PyTorch model loaded successfully")

    export_model = SigLIP2ForExport(model).eval().to(DEVICE)

    # IMPORTANT: use batch > 1 during export
    dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    onnx_path = onnx_dir / f"{onnx_name}.onnx"

    print(f"\nExporting ONNX model to: {onnx_path}")
    with torch.no_grad():
        torch.onnx.export(
            export_model,
            dummy_input,
            str(onnx_path),
            opset_version=OPSET_VERSION,
            input_names=["pixel_values"],
            output_names=["probabilities"],
            dynamic_axes={
                "pixel_values": {0: "batch_size"},
                "probabilities": {0: "batch_size"},
            },
            do_constant_folding=True,
        )

    print(f"  ✅ ONNX model exported. File size: {os.path.getsize(onnx_path) / 1e6:.0f} MB")

    # Verify ONNX structure
    try:
        onnx_model = onnx.load(str(onnx_path))
        onnx.checker.check_model(onnx_model)
        print("  ✅ ONNX model verification passed (onnx.checker)")
    except Exception as e:
        print(f"  ❌ ONNX model verification failed: {e}")
        return False

    print_onnx_shapes(onnx_path)

    # Optional runtime verification
    try:
        verify_onnxruntime(onnx_path, export_model)
    except Exception as e:
        print(f"  ⚠️ ONNX runtime verification failed: {e}")

    # Simplify after export
    sim_path = simplify_onnx(onnx_path, onnx_dir, onnx_name)

    # Thresholds
    thresh_path = config.get("thresh_path")
    thresholds = {}
    if thresh_path and Path(thresh_path).exists():
        with open(thresh_path) as f:
            thresh_data = json.load(f)
        thresholds = thresh_data.get("thresholds", {})
        print(f"  Loaded {len(thresholds)} thresholds from {thresh_path}")
    else:
        print("  ⚠️ No threshold file found, using default 0.5 for all classes.")
        thresholds = {c: 0.5 for c in classes}

    best_f1 = safe_float(ckpt.get("best_f1"), default=None)

    prod_config = {
        "model": f"SigLIP2_SO400M_{branch.title()}",
        "model_id": SIGLIP2_MODEL_ID,
        "branch": branch,
        "img_size": IMG_SIZE,
        "num_classes": num_classes,
        "class_names": classes,
        "class_to_idx": cls2idx,
        "thresholds": thresholds,
        "normalization": {
            "mean": list(NORM_MEAN),
            "std": list(NORM_STD),
        },
        "onnx": {
            "file": sim_path.name,
            "input_name": "pixel_values",
            "input_shape": ["N", 3, IMG_SIZE, IMG_SIZE],
            "output_name": "probabilities",
            "output_shape": ["N", num_classes],
            "output_includes_softmax": True,
            "dynamic_batch": True,
            "opset_version": OPSET_VERSION,
        },
        "training_info": {
            "best_f1": best_f1,
            "backbone": ckpt.get("backbone", SIGLIP2_MODEL_ID),
            "hidden_size": HIDDEN_SIZE,
            "epoch": ckpt.get("epoch", None),
        },
        "trt_conversion": {
            "fp16_command": (
                f"trtexec --onnx={sim_path.name} "
                f"--saveEngine={onnx_name}_fp16.engine "
                f"--minShapes=pixel_values:1x3x{IMG_SIZE}x{IMG_SIZE} "
                f"--optShapes=pixel_values:4x3x{IMG_SIZE}x{IMG_SIZE} "
                f"--maxShapes=pixel_values:4x3x{IMG_SIZE}x{IMG_SIZE} "
                f"--fp16 --memPoolSize=workspace:8192M"
            ),
            "fp32_command": (
                f"trtexec --onnx={sim_path.name} "
                f"--saveEngine={onnx_name}_fp32.engine "
                f"--minShapes=pixel_values:1x3x{IMG_SIZE}x{IMG_SIZE} "
                f"--optShapes=pixel_values:4x3x{IMG_SIZE}x{IMG_SIZE} "
                f"--maxShapes=pixel_values:4x3x{IMG_SIZE}x{IMG_SIZE} "
                f"--memPoolSize=workspace:8192M"
            ),
            "note": "Dynamic batch enabled. Use explicit TensorRT shape profiles.",
        },
        "production_notes": [
            "Output 'probabilities' already has softmax applied",
            "Do NOT apply softmax again — causes double-softmax bug",
            "Dynamic batch export enabled",
            "Validate TensorRT engine at batch=1 and batch=4",
        ],
    }

    config_path = onnx_dir / f"{onnx_name}_production_config.json"
    with open(config_path, "w") as f:
        json.dump(prod_config, f, indent=2)

    print(f"  ✅ Production config saved: {config_path}")
    print(f"\n  ONNX model for TRT: {sim_path}")
    print(f"  TRT FP16 Command: {prod_config['trt_conversion']['fp16_command']}")
    print(f"  TRT FP32 Command: {prod_config['trt_conversion']['fp32_command']}")
    return True


for branch, config in MODELS.items():
    convert_and_export_model(branch, config)

print("\nConversion process completed.")


######################################################################
  CONVERTING: SigLIP2 LOWER Sub-Classifier
######################################################################
Loading PyTorch checkpoint from: /content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/best_siglip2_lower.pt
  Classes (4): ['jeans', 'jogging_bottoms', 'skirts', 'trousers']
Loading SigLIP2Classifier model...


config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

  ✅ PyTorch model loaded successfully

Exporting ONNX model to: /content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/siglip2_lower.onnx


W0312 14:35:03.441000 456 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0312 14:35:03.443000 456 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0312 14:35:03.444000 456 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0312 14:35:03.445000 456 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `SigLIP2ForExport([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SigLIP2ForExport([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Applied 115 of general pattern rewrite rules.


  ✅ ONNX model exported. File size: 3 MB
  ✅ ONNX model verification passed (onnx.checker)

--- ONNX Graph I/O Shapes ---
INPUT  pixel_values: ['batch_size', 3, 384, 384]
OUTPUT probabilities: ['batch_size', 4]
  ⚠️ onnxruntime not installed. Skipping runtime verification.
  ⚠️ onnxsim not installed. Skipping simplification.
  Loaded 4 thresholds from /content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/lower_siglip2_thresholds.json
  ✅ Production config saved: /content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/siglip2_lower_production_config.json

  ONNX model for TRT: /content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_lower_sub/siglip2_lower.onnx
  TRT FP16 Command: trtexec --onnx=siglip2_lower.onnx --saveEngine=siglip2_lower_fp16.engine --minShapes=pixel_values:1x3x384x384 --optShapes=pixel_values:4x3x384x384 --maxShapes=pixel_values:4x3x384x384 --fp16 --memPoolSize=workspace:

In [ ]:
!pip install onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 20.8 MB/s eta 0:00:00
